<a href="https://colab.research.google.com/github/yoshiagent/mathModel-colab/blob/main/notebooks/Mech_Elec_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 機械與電路系統的建模：轉動慣量、RC／RL、MCK、RCL

對照網頁：[機械系統基礎](https://yoshiagent.github.io/mathModel/ME.html)、[電路系統基礎](https://yoshiagent.github.io/mathModel/EL.html)、
[RC 與 RL](https://yoshiagent.github.io/mathModel/RCRL.html)、[MCK](https://yoshiagent.github.io/mathModel/MCK.html)、[RCL](https://yoshiagent.github.io/mathModel/RCL.html)

| 原檔／原講義 | 內容 | 本筆記本 |
|---|---|---|
| Moment of Inertia | 圓柱、空心圓柱、長方體、平行軸定理（以積分驗證） | 第 1 節 |
| Equivalent Moment of Inertia | 齒輪系統、直線運動的等效轉動慣量 | 第 2 節 |
| `RCData.m`、`RC.mdl` | RC 電路的 Simulink 模型（積分器＋回授） | 第 3 節 |
| RC／RL 高通與低通 | 四種一階電路的步階響應 | 第 4 節 |
| `mck.mdl` | MCK 微分方程式方塊圖、方波輸入 | 第 5 節 |
| `mckss.m`、`ssmodel.mdl`、`SysTF=tf(SysSS)` | 狀態空間與轉換成轉移函數 | 第 6 節 |
| `MCKTFM.m`、`MCKTf.mdl` | 轉移函數、$\omega_n$、$\xi$ | 第 6 節 |
| `RCLData.m`、`RCL.mdl` | RCL 電路與 MCK 的對應 | 第 7 節 |

只用 Colab 內建的 numpy、scipy、sympy、plotly。

In [1]:
import numpy as np
import sympy as sp
from scipy import signal
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show(fig, title, xlab=None, ylab=None, h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()

## 1. 常見形狀的轉動慣量（以積分驗證網頁上的公式）

$J = \int r^2\,dm$。用 sympy 對均質圓柱、空心圓柱、長方體積分，與網頁公式比較（$m$ 為質量、$\rho$ 為密度）。

In [2]:
r, th, z, x, y = sp.symbols("r theta z x y", real=True)
rho, L, D1, D2, A, B, C, l = sp.symbols("rho L D_1 D_2 A B C l", positive=True)

# 實心圓柱（直徑 D1、長 L）：x 為中心軸，y 為通過中心、垂直中心軸的軸
m_cyl = rho * sp.pi * D1 ** 2 / 4 * L
Jx = sp.integrate(r ** 2 * rho * r, (r, 0, D1 / 2), (th, 0, 2 * sp.pi), (z, -L / 2, L / 2))
Jy = sp.integrate(((r * sp.cos(th)) ** 2 + z ** 2) * rho * r, (r, 0, D1 / 2), (th, 0, 2 * sp.pi), (z, -L / 2, L / 2))
print("cylinder  Jx - m D1^2/8              =", sp.simplify(Jx - m_cyl * D1 ** 2 / 8), ";  Jx =", sp.factor(Jx))
print("cylinder  Jy - m/4 (D1^2/4 + L^2/3)  =", sp.simplify(Jy - m_cyl / 4 * (D1 ** 2 / 4 + L ** 2 / 3)))

# 空心圓柱（外徑 D1、內徑 D2）
m_h = rho * sp.pi * (D1 ** 2 - D2 ** 2) / 4 * L
Jxh = sp.integrate(r ** 2 * rho * r, (r, D2 / 2, D1 / 2), (th, 0, 2 * sp.pi), (z, -L / 2, L / 2))
Jyh = sp.integrate(((r * sp.cos(th)) ** 2 + z ** 2) * rho * r, (r, D2 / 2, D1 / 2), (th, 0, 2 * sp.pi), (z, -L / 2, L / 2))
print("hollow    Jx - m (D1^2+D2^2)/8        =", sp.simplify(Jxh - m_h * (D1 ** 2 + D2 ** 2) / 8))
print("hollow    Jx - pi/32 rho L (D1^4-D2^4) =", sp.simplify(Jxh - sp.pi / 32 * rho * L * (D1 ** 4 - D2 ** 4)))
print("hollow    Jy - m/4((D1^2+D2^2)/4+L^2/3)=", sp.simplify(Jyh - m_h / 4 * ((D1 ** 2 + D2 ** 2) / 4 + L ** 2 / 3)))

# 長方體（A × B × C），x 軸平行 C 邊
m_b = rho * A * B * C
Jxb = sp.integrate((x ** 2 + y ** 2) * rho, (x, -A / 2, A / 2), (y, -B / 2, B / 2), (z, -C / 2, C / 2))
print("pillar    Jx - m (A^2+B^2)/12         =", sp.simplify(Jxb - m_b * (A ** 2 + B ** 2) / 12))
# 平行軸：繞距離質心 l 的平行軸
Jxl = sp.integrate(((x - l) ** 2 + y ** 2) * rho, (x, -A / 2, A / 2), (y, -B / 2, B / 2), (z, -C / 2, C / 2))
print("parallel  Jx - m (A^2+B^2+12 l^2)/12  =", sp.simplify(Jxl - m_b * (A ** 2 + B ** 2 + 12 * l ** 2) / 12))

cylinder  Jx - m D1^2/8              = 0 ;  Jx = pi*D_1**4*L*rho/32
cylinder  Jy - m/4 (D1^2/4 + L^2/3)  = 0
hollow    Jx - m (D1^2+D2^2)/8        = 0
hollow    Jx - pi/32 rho L (D1^4-D2^4) = 0
hollow    Jy - m/4((D1^2+D2^2)/4+L^2/3)= 0
pillar    Jx - m (A^2+B^2)/12         = 0
parallel  Jx - m (A^2+B^2+12 l^2)/12  = 0


## 2. 等效轉動慣量

**齒輪系統**：速比 $n = \omega_2/\omega_1 = r_1/r_2 = N_1/N_2$，馬達端看到的 $I_{eq} = I_1 + n^2I_2$（能量法：$\frac12 I_2\omega_2^2 = \frac12 I_{2e}\omega_1^2$）。
用兩個齒輪的運動方程式（接觸力 $F$）直接模擬，和 $I_{eq}$ 的單一慣量模型比較。

In [3]:
I1, I2, r1, r2, T = 0.002, 0.05, 0.02, 0.08, 0.1         # kg m^2, m, N m（例）
n = r1 / r2
# 兩齒輪：T - F r1 = I1 a1，F r2 = I2 a2，a2 = n a1 → 解出 a1
a1 = T / (I1 + I2 * (r1 / r2) * n)
print(f"n = {n}, alpha1 (two-gear equations) = {a1:.4f} rad/s^2, alpha1 = T/Ieq = {T / (I1 + n ** 2 * I2):.4f} rad/s^2, Ieq = {I1 + n ** 2 * I2:.5f}")
print(f"kinetic energy check: 1/2 I1 w1^2 + 1/2 I2 (n w1)^2 = {0.5 * I1 + 0.5 * I2 * n ** 2:.5f} = 1/2 Ieq (w1 = 1)")

n = 0.25, alpha1 (two-gear equations) = 19.5122 rad/s^2, alpha1 = T/Ieq = 19.5122 rad/s^2, Ieq = 0.00513
kinetic energy check: 1/2 I1 w1^2 + 1/2 I2 (n w1)^2 = 0.00256 = 1/2 Ieq (w1 = 1)


**直線運動的物體**（例如滾珠螺桿帶動的平台）：$\frac12 mv^2 = \frac12 J\omega^2 \Rightarrow J = m\left(\dfrac{v}{\omega}\right)^2 = m\left(\dfrac{A}{2\pi}\right)^2$，$A$ 為每轉一圈的移動量。

In [4]:
m_table, lead = 20.0, 0.01                                 # 20 kg、導程 10 mm
J_eq = m_table * (lead / (2 * np.pi)) ** 2
print(f"J = m (A/2pi)^2 = {J_eq:.3e} kg m^2  (m = {m_table} kg, A = {lead * 1000:.0f} mm/rev)")

J = m (A/2pi)^2 = 5.066e-05 kg m^2  (m = 20.0 kg, A = 10 mm/rev)


## 3. RC 電路（`RCData.m`、`RC.mdl`）

$RC\dfrac{dv_C}{dt} + v_C = u_s(t)$，$v_C(0) = 0$。`RC.mdl` 把它畫成方塊圖：
Step（t = 0 跳到 1）→ Gain $1/(RC)$ → Sum（＋ −）→ Integrator → $v_C$，$v_C$ 經 Gain1 $1/(RC)$ 回授到 Sum 的負端；模擬 10 s。
`RCData.m`：`R=1; C=1;`。這個方塊圖其實就是 $\dot v_C = \frac{1}{RC}(u - v_C)$。

In [5]:
R, Cc = 1.0, 1.0
t = np.linspace(0, 10, 1001)
sol = solve_ivp(lambda tt, v: [(1.0 - v[0]) / (R * Cc)], (0, 10), [0.0], t_eval=t, rtol=1e-10, atol=1e-12)
vc = sol.y[0]
print(f"vC(RC) = {np.interp(R * Cc, t, vc):.4f} (1 - e^-1 = {1 - np.exp(-1):.4f}),  vC(10) = {vc[-1]:.5f}  (Display block)")
print(f"max |block diagram - (1 - exp(-t/RC))| = {np.max(np.abs(vc - (1 - np.exp(-t / (R * Cc))))):.1e}")
fig = go.Figure()
fig.add_scatter(x=t, y=np.round(vc, 5), name="vC (RC.mdl)")
show(fig, "RC.mdl: R = C = 1, step input", "t (s)", "vC (V)")

vC(RC) = 0.6321 (1 - e^-1 = 0.6321),  vC(10) = 0.99995  (Display block)
max |block diagram - (1 - exp(-t/RC))| = 2.0e-11


## 4. 四種一階電路

| 電路 | 輸出 | 轉移函數 | 類型 |
|---|---|---|---|
| RC | $v_C$ | $\dfrac{1}{RCs+1}$ | 低通 |
| RC | $v_R$ | $\dfrac{RCs}{RCs+1}$ | 高通 |
| RL | $v_R$ | $\dfrac{1}{\frac{L}{R}s+1}$ | 低通 |
| RL | $v_L$ | $\dfrac{\frac{L}{R}s}{\frac{L}{R}s+1}$ | 高通 |

以 $R = 1\,\Omega$、$C = 1$ F、$L = 2$ H 為例：兩個低通的步階響應從 0 升到 1，兩個高通從 1 衰減到 0；
任一電路的兩個輸出相加恆等於輸入（KVL）。

In [6]:
Lh = 2.0
cases = {"RC: vC (low-pass)": ([1], [R * Cc, 1]), "RC: vR (high-pass)": ([R * Cc, 0], [R * Cc, 1]),
         "RL: vR (low-pass)": ([1], [Lh / R, 1]), "RL: vL (high-pass)": ([Lh / R, 0], [Lh / R, 1])}
t = np.linspace(0, 10, 501)
fig = go.Figure()
out = {}
for name, (num, den) in cases.items():
    _, yy = signal.step(signal.lti(num, den), T=t)
    out[name] = yy
    fig.add_scatter(x=t, y=np.round(yy, 5), name=name)
show(fig, "Step responses of RC and RL circuits (R = 1, C = 1, L = 2)", "t (s)", "V")
print(f"KVL check RC: max |vC + vR - 1| = {np.max(np.abs(out['RC: vC (low-pass)'] + out['RC: vR (high-pass)'] - 1)):.1e};  "
      f"RL: max |vR + vL - 1| = {np.max(np.abs(out['RL: vR (low-pass)'] + out['RL: vL (high-pass)'] - 1)):.1e}")
print(f"RL time constant L/R = {Lh / R} s: vR(L/R) = {np.interp(Lh / R, t, out['RL: vR (low-pass)']):.4f}")

KVL check RC: max |vC + vR - 1| = 0.0e+00;  RL: max |vR + vL - 1| = 0.0e+00
RL time constant L/R = 2.0 s: vR(L/R) = 0.6321


## 5. MCK 的微分方程式方塊圖（`mck.mdl`）

$\ddot x = \frac{1}{m}f(t) - \frac{c}{m}\dot x - \frac{k}{m}x$：Signal Generator → Gain $1/m$ → Sum（$-\,+\,-$）→ 兩個積分器 → $x$，
再以 Gain1 $c/m$、Gain2 $k/m$ 回授。Signal Generator 設定為**方波、振幅 1、頻率 0.1 Hz**（週期 10 s，前 5 s 為 +1、後 5 s 為 −1），模擬 10 s。
網頁沒有給 m、c、k，這裡和 `mckss.m` 一樣取 $m = c = k = 1$。

In [7]:
m, c, k = 1.0, 1.0, 1.0
f = lambda tt: 1.0 if (tt % 10) < 5 else -1.0                         # 0.1 Hz 方波：前半週期 +1、後半 -1
t = np.linspace(0, 10, 2001)
rhs = lambda tt, s, u: [s[1], u / m - c / m * s[1] - k / m * s[0]]
s1 = solve_ivp(rhs, (0, 5), [0, 0], args=(1.0,), dense_output=True, rtol=1e-11, atol=1e-13)      # 方波在 t = 5 跳變：分兩段積分
s2 = solve_ivp(rhs, (5, 10), s1.y[:, -1], args=(-1.0,), dense_output=True, rtol=1e-11, atol=1e-13)
x5 = np.where(t <= 5, s1.sol(np.minimum(t, 5))[0], s2.sol(np.maximum(t, 5))[0])
print(f"x(5) = {np.interp(5, t, x5):.4f}, max x = {x5.max():.4f} at t = {t[np.argmax(x5)]:.2f} s, x(10) = {x5[-1]:.4f}")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("f(t): square wave 0.1 Hz", "x(t)"))
fig.add_scatter(x=t, y=[f(tt) for tt in t], name="f", row=1, col=1)
fig.add_scatter(x=t, y=np.round(x5, 5), name="x", row=2, col=1)
fig.update_xaxes(title="t (s)", row=2, col=1)
show(fig, "mck.mdl: m = c = k = 1, square-wave force", h=520)

x(5) = 1.0746, max x = 1.1630 at t = 3.63 s, x(10) = -1.1470


## 6. 狀態空間與轉移函數（`mckss.m`、`ssmodel.mdl`、`MCKTFM.m`、`MCKTf.mdl`）

```matlab
m=1; c=1; k=1;
A=[0 1;-k/m -c/m]; B=[0;1/m]; C=[1 0]; D=0;
SysSS=ss(A,B,C,D);
step(SysSS)
SysTF=tf(SysSS)          % 轉換成轉移函數
MCKTF=tf([1],[M C K]);   % MCKTFM.m
step(MCKTF)
```
`ssmodel.mdl` 是同一個狀態空間方塊，輸入和 `mck.mdl` 一樣是 0.1 Hz 方波。

In [8]:
A_ = np.array([[0, 1], [-k / m, -c / m]])
B_ = np.array([[0], [1 / m]])
C_ = np.array([[1, 0]])
D_ = np.array([[0]])
num, den = signal.ss2tf(A_, B_, C_, D_)
print("SysTF = tf(SysSS): num =", np.round(num, 10), " den =", np.round(den, 10))
wn, zeta = np.sqrt(k / m), c / (2 * np.sqrt(m * k))
print(f"wn = sqrt(k/m) = {wn}, zeta = c/(2 sqrt(mk)) = {zeta}")

t = np.linspace(0, 12, 1201)
_, y_ss = signal.step(signal.StateSpace(A_, B_, C_, D_), T=t)
_, y_tf = signal.step(signal.lti([1], [m, c, k]), T=t)
print(f"max |step(SysSS) - step(MCKTF)| = {np.max(np.abs(y_ss - y_tf)):.1e};  peak = {y_tf.max():.4f} at t = {t[np.argmax(y_tf)]:.2f} s")
_, y_sq, _ = signal.lsim(signal.StateSpace(A_, B_, C_, D_), [f(tt) for tt in np.linspace(0, 10, 2001)], np.linspace(0, 10, 2001), interp=False)   # 零階保持＝方波在取樣點之間不變
print(f"ssmodel.mdl vs mck.mdl (square wave): max difference = {np.max(np.abs(y_sq - x5)):.1e}")
fig = go.Figure()
fig.add_scatter(x=t, y=np.round(y_ss, 5), name="step(SysSS)")
fig.add_scatter(x=t, y=np.round(y_tf, 5), name="step(MCKTF)", line=dict(dash="dash"))
show(fig, "MCK step response: state space = transfer function (m = c = k = 1)", "t (s)", "x")

SysTF = tf(SysSS): num = [[0. 0. 1.]]  den = [1. 1. 1.]
wn = sqrt(k/m) = 1.0, zeta = c/(2 sqrt(mk)) = 0.5
max |step(SysSS) - step(MCKTF)| = 2.2e-16;  peak = 1.1630 at t = 3.63 s
ssmodel.mdl vs mck.mdl (square wave): max difference = 3.3e-12


## 7. RCL 電路（`RCLData.m`、`RCL.mdl`）

$L\ddot q + R\dot q + \frac{1}{C}q = v_i$，$v_c = q/C$：$\dfrac{V_c(s)}{V_i(s)} = \dfrac{1}{LCs^2 + RCs + 1}$，
$\omega_n = 1/\sqrt{LC}$、$\xi = \frac{R}{2}\sqrt{C/L}$。`RCLData.m`：`L=1; C=1; R=1;`，和 $m = c = k = 1$ 的 MCK 是同一個系統。
對應關係：$L \leftrightarrow m$、$R \leftrightarrow c$、$1/C \leftrightarrow k$、$q \leftrightarrow x$、$v_i \leftrightarrow f$。

In [9]:
Lr, Rr, Cr = 1.0, 1.0, 1.0
_, y_rcl = signal.step(signal.lti([1], [Lr * Cr, Rr * Cr, 1]), T=t)
print(f"wn = {1 / np.sqrt(Lr * Cr)}, zeta = {Rr / 2 * np.sqrt(Cr / Lr)},  max |RCL - MCK| = {np.max(np.abs(y_rcl - y_tf)):.1e}")
print(f"overshoot = {100 * (y_rcl.max() - 1):.1f} %  (exp(-zeta pi/sqrt(1-zeta^2)) = {100 * np.exp(-0.5 * np.pi / np.sqrt(0.75)):.1f} %)")
fig = go.Figure()
for Rv in (0.5, 1.0, 2.0, 3.0):
    _, yy = signal.step(signal.lti([1], [Lr * Cr, Rv * Cr, 1]), T=t)
    fig.add_scatter(x=t, y=np.round(yy, 5), name=f"R = {Rv} (zeta = {Rv / 2:.2f})")
show(fig, "RCL circuit (L = C = 1): effect of R on the capacitor voltage", "t (s)", "vc (V)")

wn = 1.0, zeta = 0.5,  max |RCL - MCK| = 0.0e+00
overshoot = 16.3 %  (exp(-zeta pi/sqrt(1-zeta^2)) = 16.3 %)
